# 11. PSMA phantoms: one phantom per study

Builds `data/processed/psma/phantoms/<study_id>/` for every usable study of PSMA-PET-CT-Lesions, on the PET grid:

| File | Content |
|---|---|
| `ct.nii.gz` | CT in HU (int16) |
| `pet_pct_ia_ml.nii.gz` | PET in decay-corrected %IA/mL = SUV x 100 / W_g |
| `organs.nii.gz` | one integer label per compartment, map in `phantoms/labelmap.json` |
| `lesions.nii.gz` | lesions of `labelsTr` as 26-connected components, numbered from 1 |
| `meta.json` | tracer, weight, activity, uptake time, contrast, volumes, masses, mean and total %IA, lesions, QC flags |

Code: `src/psma_data.py` (single loader, applies the exclusion list and the crop of `src/psma_exclusions.py`) and `src/psma_phantoms.py` (masks, statistics, writing). A phantom folder appears only when complete, so the build can be resumed.

**Label design.** Labels are exclusive, so three requests need more than one integer:
- *kidneys* = 1 functional + 2 non-functional (fluid): total = 1 + 2;
- *salivary glands* = 5-8, one label per gland, because the PBPK flag is per gland;
- *spine* = 12 L2-L4 (red-marrow surrogate) + 13 the rest (C1-L5, S1, sacrum): whole spine = 12 + 13.

The other labels: 3 liver, 4 spleen, 9 bladder, 10 GI tract (stomach, duodenum, small bowel, colon, single label; parts in `meta.json`), 11 blood pool (TotalSegmentator aorta eroded by one voxel with 6-connectivity; the removed wall goes to rest of body), 14 rest of body, 15 lesion. Lesions overwrite every organ; the overlap per compartment is stored. Heart is not a compartment here, so it is in rest of body.

**Masses.** Contrast raises the HU of kidneys, liver, spleen and vessels, so the CT cannot give their density. Soft organs use the ICRP 110 Table A.1 adult male densities (read from `AM_organs.dat`): kidney 1.05, liver 1.05, spleen 1.04, salivary glands 1.03, urine and bladder 1.04, blood 1.06 g/mL. GI tract (gas in the lumen), spine and rest of body (lungs, fat, bone) use the Schneider 2000 table of the GATE phantom, with HU clipped at +50 for the GI tract against oral and IV contrast. Lesions contribute by the Schneider density to the total body mass only.

**About this notebook.** One phantom per usable PSMA study: CT, PET in %IA/mL, compartment labels, lesions; QC and biodistribution check.
- **Inputs:** raw PSMA data, segmentations, `pet_dicom_headers.csv`; `src/psma_data.py`, `src/psma_phantoms.py`.
- **Outputs:** `data/processed/psma/phantoms/`, `phantoms_summary.csv`, `phantoms_qc/`.
- **Run time:** about 1 min with the phantoms built; the build takes hours.
- **Environment:** .venv.

In [ ]:
import json, sys
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

WORKSPACE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WORKSPACE / "src"))
import psma_data as data
import psma_phantoms as ph

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", lambda v: f"{v:.3g}")
plt.rcParams.update({"figure.dpi": 90, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
COHORT, LITERATURE, INK = "#2a78d6", "#eb6834", "#52514e"
QC = data.PROCESSED / "phantoms_qc"
QC.mkdir(exist_ok=True)

table = data.study_table()
inventory = pd.read_csv(data.INVENTORY)
print(f"{len(inventory)} studies in the inventory, {len(table)} usable after the exclusion list")
print(table.groupby("tracer").size().to_dict(), "| PBPK cohort:", int(table["pbpk_cohort"].sum()))
try:
    data.load_study(next(iter(data.SUV_EXCLUSIONS)), table)
except ValueError as error:
    print("loader refuses excluded studies:", str(error)[:90])

## 1. Build

Resumable: existing phantoms are skipped. About 10-30 s per study per process.

In [ ]:
log = ph.build_all(table["study_id"], processes=6)
log.to_csv(QC / "build_log_last_run.csv", index=False)
print(log["status"].value_counts().to_dict(), "| built in", round(log.loc[log["status"] == "built", "seconds"].sum() / 3600, 2), "process-hours")
print(log.loc[log["status"] == "error", ["study_id", "error"]].to_string())

summary = ph.summary_table()
summary.to_csv(data.PROCESSED / "phantoms_summary.csv", index=False)
print(summary.shape)
print("disk:", round(sum(f.stat().st_size for f in ph.PHANTOMS.rglob("*") if f.is_file()) / 1e9, 1), "GB")

## 2. Consistency checks

Every voxel of the body has one label, the totals add up, the lesion numbering agrees with the inventory of notebook 08, and the files on disk give back the source SUV.

In [ ]:
metas = {p.parent.name: json.loads(p.read_text()) for p in sorted(ph.PHANTOMS.glob("psma_*/meta.json"))}
assert set(metas) == set(table["study_id"]), "missing phantoms"

gap = []
for sid, m in metas.items():
    parts = sum(c["total_pct_ia"] for c in m["compartments"].values()) + m["lesions"]["total_pct_ia"]
    gap.append(abs(parts - m["totals"]["body_pct_ia"]))
print("largest |sum of compartments - body %IA| over all studies:", round(max(gap), 4), "%IA")

check = summary[["study_id", "n_lesions", "lesions_volume_ml"]].merge(inventory[["study_id", "n_lesions", "tlv_ml"]], on="study_id", suffixes=("", "_inv"))
print("lesion count differs from nb08:", check.loc[check["n_lesions"] != check["n_lesions_inv"], "study_id"].tolist())
print("largest |TLV - nb08|:", round((check["lesions_volume_ml"] - check["tlv_ml"]).abs().max(), 3), "mL")

rng = np.random.default_rng(20261103)
for sid in list(rng.choice(table["study_id"], 4, replace=False)) + ["psma_3959d1c381a5bcd6_2015-11-12"]:
    folder = ph.PHANTOMS / sid
    pet, organs, lesions = (nib.load(folder / f) for f in ("pet_pct_ia_ml.nii.gz", "organs.nii.gz", "lesions.nii.gz"))
    source = nib.load(data.IMAGES / f"{sid}_0001.nii.gz")
    k = metas[sid]["grid"]["k_min_of_original"]
    suv = np.asanyarray(source.dataobj)[:, :, k:]
    ratio = np.asanyarray(pet.dataobj).sum() / suv.sum() / (100 / (metas[sid]["weight_kg"] * 1000))
    world = source.affine @ np.array([0, 0, k, 1]) - pet.affine @ np.array([0, 0, 0, 1])
    l, o = np.asanyarray(lesions.dataobj), np.asanyarray(organs.dataobj)
    print(sid[5:], "PET/(SUV 100/W_g) =", round(float(ratio), 5), "| origin shift mm:", np.abs(world[:3]).max().round(3),
          "| lesion voxels with label 15:", bool(np.all(o[l > 0] == 15)), "| label 15 without lesion:", int(np.sum((o == 15) & (l == 0))))

### Body mask and activity outside it

Threshold at -300 HU, in-plane opening of 6 mm to detach the couch, largest 3D component plus other components above 50 mL whose PET is not empty (the couch has no activity), slice-wise hole filling. A slice below 0.8 of the median area of the 15 slices around it has an open chest wall (its lungs stay unfilled): it is closed in-plane by 15 mm and filled again. Masks of organs and lesions are added on top. The mass inside the FOV is a plausibility check against the header weight.

In [ ]:
tot = summary.drop(summary.index[summary["flag__cropped_not_whole_body"]])
print(tot[["totals__fov_pct_ia", "totals__outside_body_pct_ia", "totals__body_mass_over_weight"]].describe(percentiles=[.05, .5, .95]).T)
print("studies with > 5 %IA outside the body mask:", tot["flag__pet_outside_body"].sum())
print(tot.sort_values("totals__outside_body_pct_ia")[["study_id", "tracer", "totals__fov_pct_ia", "totals__outside_body_pct_ia"]].tail(6).to_string(index=False))
over = tot.assign(group=pd.cut(tot["totals__fov_pct_ia"], [0, 100, 110, 120, 1000], labels=["<=100", "100-110", "110-120", ">120"]))
print(over.groupby(["tracer", "group"], observed=False).size().unstack().to_string())
print(tot.loc[tot["totals__fov_pct_ia"] > 110, ["study_id", "tracer", "scanner", "weight_kg", "injected_activity_mbq", "uptake_time_min", "totals__fov_pct_ia"]].to_string(index=False))
print(tot.groupby(["tracer", "scanner"]).agg(n=("study_id", "size"), fov_pct_ia=("totals__fov_pct_ia", "median"), uptake_min=("uptake_time_min", "median"), liver_suv=("liver__mean_suv", "median"), rest_suv=("rest_of_body__mean_suv", "median"), mass_over_weight=("totals__body_mass_over_weight", "median")).round(2).to_string())
print(tot.sort_values("totals__pct_ia_in_slices_without_body")[["study_id", "tracer", "totals__slices_without_body", "totals__pct_ia_in_slices_without_body", "urinary_bladder__volume_ml"]].tail(4).to_string(index=False))
print("mass in FOV / weight outside 0.5-1.0:", tot.loc[~tot["totals__body_mass_over_weight"].between(.5, 1.0), ["study_id", "weight_kg", "totals__body_mass_kg"]].to_string(index=False))

**Reading.** All 589 phantoms close: compartments plus lesions give the body %IA, lesions match nb08, the files give back the source SUV.

- **Injected activity in the FOV.** Median 75.9 %IA for 68Ga (range 48-97, none above 100) and 90.2 %IA for 18F. A decay-corrected total above 100 is not physical: 39 18F studies exceed it, 38 of them by less than 10 points, which is in the range of a calibration or activity-measurement error.
- **`psma_54d1613921a2d2ae_2021-02-01` (18F) was excluded after this check,** so it is no longer in the tables: 143 %IA in the FOV, with an uptake time of 132 min against a median of about 61 min on that scanner. With 61 min the decay factor falls by 1.57 and the total becomes about 91 %IA, the typical 18F value, so the header time is likely wrong, but this is not verifiable. The other study of the subject stays. `psma_18eba3b35ee1ddac_2020-09-05` (112 %IA, uptake 79 min) has no such explanation and stays.
- **Scanner effect, unresolved.** Median FOV %IA ranges from 71.5 (Discovery690) to 82.6 (Biograph64) for 68Ga, and from 81 to 93 for 18F. Uptake time also differs (71 vs 61 min for 68Ga), so voiding, uptake and calibration cannot be separated here. Keep scanner as a covariate in the PBPK fit.
- **Outside the body.** Median 1.7 %IA, 6 studies above 5. On the 3 viewed, the mask follows the patient, with two exceptions. `psma_81b53d3911e09543_2021-03-13` (18F) has a CT that stops 68 slices before the PET: 5.7 %IA and the bladder are outside the masks. Subject `41260c3678449a2f` has an open chest wall on some slices: the repair moved up to 0.75 %IA back into the body. After it, 6 studies still have collapsed slices, holding at most 0.43 %IA outside the mask, declared and not fixed further. The other 3 of the 6 were not viewed.
- **Mass in the FOV / weight.** Median 0.68, as expected without head top and legs. Two studies fall below 0.5 (`psma_35fd083826d0b23d_2018-03-23`, `psma_bbbe1ae9a074871b_2014-06-27`), possibly wrong weights, not verified. A wrong weight cancels in %IA.

## 3. Functional kidney

**Method.** Inside each TotalSegmentator kidney: 3x3x3 median of the CT using only kidney voxels (outside voxels take the nearest kidney value, so fat does not enter the filter; the CT at PET resolution is noisy), then voxels below a threshold are fluid. Kidney rim voxels are partial-volume mixtures with perirenal fat and would peel the whole surface at any threshold above -20 HU, so a rim voxel counts only when it touches an interior fluid voxel. Components smaller than a minimum volume are noise. Fluid voxels = label 2; the rest = label 1, stored with the total per side in `meta.json`, together with the functional volume at thresholds 0-40 HU.

**CT type comes from the images, not only from the metadata.** A study is *enhanced* if the larger kidney parenchyma (median HU in the eroded mask) is at least 60 HU.

In [ ]:
long = []
for side in ("left", "right"):
    part = summary[["study_id", "subject", "tracer", "ct_contrast_metadata", "ct_kidney_enhanced"]].copy()
    part["side"] = side
    for c in summary.columns:
        if c.startswith(f"kidney_{side}__"):
            part[c.split("__")[1]] = summary[c]
    long.append(part)
kidneys = pd.concat(long, ignore_index=True)
kidneys = kidneys[kidneys["total_ml"] >= 50].copy()
kidneys["nonfunctional_fraction"] = kidneys["nonfunctional_ml"] / kidneys["total_ml"]

print(kidneys.groupby("ct_kidney_enhanced")["parenchyma_hu"].describe(percentiles=[.05, .5, .95]).round(1))
print("\nStudies where metadata and images disagree:")
print(summary.loc[summary["ct_contrast_metadata"] != summary["ct_kidney_enhanced"], ["study_id", "tracer", "ct_contrast_metadata", "ct_kidney_enhanced", "kidney_left__parenchyma_hu", "kidney_right__parenchyma_hu"]].to_string(index=False))
print("\nEnhanced / unenhanced studies:", summary["ct_kidney_enhanced"].value_counts().to_dict())

thr = [0, 5, 10, 15, 20, 30, 40]
enhanced = kidneys[kidneys["ct_kidney_enhanced"]]
fractions = pd.DataFrame({t: 1 - enhanced[f"functional_ml_thr{t}"] / enhanced["total_ml"] for t in thr})
print("\nEnhanced CTs, fraction of kidney below threshold, percentiles over kidneys:")
print(fractions.describe(percentiles=[.5, .9, .95, .99]).loc[["mean", "50%", "90%", "95%", "99%", "max"]].round(4))
withfluid = enhanced[enhanced["nonfunctional_ml"] >= 5]
print(f"\nKidneys with >= 5 mL of fluid at 20 HU: {len(withfluid)} of {len(enhanced)}")
steps = pd.DataFrame({f"{a}->{b}": (withfluid[f"functional_ml_thr{a}"] - withfluid[f"functional_ml_thr{b}"]) / withfluid["nonfunctional_ml"] for a, b in zip(thr[:-1], thr[1:])})
print("Median change of the fluid volume per threshold step, as a fraction of the fluid volume at 20 HU:")
print(steps.median().round(3).to_string())
ratio = enhanced["fluid_mean_pct_ia_ml"] / enhanced["functional_mean_pct_ia_ml"]
print("\nPET in the fluid voxels / functional voxels, enhanced CTs, median and IQR:", ratio.quantile([.25, .5, .75]).round(2).tolist())

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for _, row in withfluid.iterrows():
    ax.plot(thr, [(row["total_ml"] - row[f"functional_ml_thr{t}"]) / row["nonfunctional_ml"] for t in thr], color=COHORT, lw=.7, alpha=.35)
ax.plot(thr, [np.median((withfluid["total_ml"] - withfluid[f"functional_ml_thr{t}"]) / withfluid["nonfunctional_ml"]) for t in thr], color=INK, lw=2, label="median")
ax.axvline(20, color=LITERATURE, lw=1.2, label="chosen threshold")
ax.set(xlabel="Threshold on the smoothed CT (HU)", ylabel="Fluid volume / fluid volume at 20 HU", ylim=(0, 1.8),
       title=f"Enhanced CTs: fluid volume grows half as fast above 20 HU ({len(withfluid)} kidneys)")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(QC / "kidney_threshold_enhanced.png", dpi=150); plt.show()

In [ ]:
def kidney_panel(study_ids_sides, path, window=(-100, 250), slices=6):
    rows = len(study_ids_sides)
    fig, axes = plt.subplots(rows * 2, slices, figsize=(slices * 1.9, rows * 3.8), squeeze=False)
    for r, (fragment, side) in enumerate(study_ids_sides):
        sid = next(s for s in metas if fragment in s)
        study = data.load_study(sid, table)
        kidney = study.organs == (1 if side == "left" else 2)
        organs = np.asanyarray(nib.load(ph.PHANTOMS / sid / "organs.nii.gz").dataobj)
        fluid = kidney & (organs == 2)
        box = tuple(np.where(kidney.any(axis=tuple(a for a in range(3) if a != i)))[0] for i in range(3))
        b = tuple(slice(max(0, x[0] - 6), x[-1] + 7) for x in box[:2])
        vmax = np.percentile(study.pet_pct_ia_ml[kidney], 98)
        k_info = metas[sid]["kidneys"][side]
        for j, k in enumerate(np.linspace(box[2][1], box[2][-2], slices).astype(int)):
            ax = axes[2 * r, j]
            ax.imshow(study.ct_hu[b + (k,)].T, cmap="gray", vmin=window[0], vmax=window[1], origin="lower")
            ax.contour(kidney[b + (k,)].T, [.5], colors="#eda100", linewidths=.8)
            if fluid[b + (k,)].any():
                ax.contour(fluid[b + (k,)].T, [.5], colors="#e34948", linewidths=1)
            ax.set_axis_off()
            if j == 0:
                ax.set_title(f"{sid[5:13]} {sid[-10:]} {side}: total {k_info['total_ml']:.0f}, functional {k_info['functional_ml']:.0f} mL", fontsize=8, loc="left", x=0)
            ax = axes[2 * r + 1, j]
            ax.imshow(study.pet_pct_ia_ml[b + (k,)].T, cmap="gray_r", vmin=0, vmax=vmax, origin="lower")
            ax.contour(kidney[b + (k,)].T, [.5], colors="#2a78d6", linewidths=.8)
            if fluid[b + (k,)].any():
                ax.contour(fluid[b + (k,)].T, [.5], colors="#e34948", linewidths=1)
            ax.set_axis_off()
    fig.tight_layout(); fig.savefig(path, dpi=130); plt.show()

kidney_panel([("e51e9429c79eff86", "left"), ("e51e9429c79eff86", "right"), ("f46740fcc3f44f10_2020-01", "left"), ("313ff961c59ee5dc_2016", "right"), ("dc21e961e5341e6b", "left"), ("35852148926fc57d", "left")], QC / "kidney_contrast_cases.png")

**Contrast CTs: reading and decision.** Each pair of rows is CT, then PET on the same slices (yellow or blue = TotalSegmentator kidney, red = fluid). 

**Decision: 20 HU, minimum component 0.25 mL, kept.** The curve does not reach a true plateau. The growth of the fluid volume per 5 HU falls from 0.20-0.25 below 20 HU to 0.13 and 0.08 above it, so 20 HU is the knee and the result is mildly threshold-dependent (median fluid volume 0.8 at 15 HU, 1.27 at 30 HU, relative to 20 HU).

- The panels agree: the cysts of `e51e9429` (left, 419 -> 218 mL), `f46740fc` and `313ff961` are removed and cold on the PET, `dc21e961` loses its sinus, `35852148` keeps almost everything.
- At cohort level the median kidney loses nothing (95th percentile 1.5%, 42 of 1121 kidneys have at least 5 mL of fluid). The PET in removed voxels is 0.29 (IQR 0.17-0.44) of the functional PET, consistent with cold fluid plus spill-in.
- The panels also show the PET cortex shifted by about one voxel from the CT mask, which section 5 quantifies.
- Metadata and images disagree in 2 studies (`psma_af881eca29658895_2020-05-09`, `psma_d0f877ac02b84ee6_2014-10-30`): parenchyma 80-210 HU with metadata saying no contrast. The image decision is used.

### Non-contrast CTs

Parenchyma has no enhancement (about 26 HU against 172 HU), so the fluid-parenchyma gap is 20-25 HU and comparable with the voxel noise: a threshold of 20 HU would remove about a third of a normal kidney. The operating point on these CTs is therefore **10 HU on the smoothed CT with a minimum component of 2 mL**. The checks below: the volume at each threshold, the PET in the removed voxels, and subjects who also have a contrast CT.

In [ ]:
plain = kidneys[~kidneys["ct_kidney_enhanced"]].copy()
print(len(plain), "kidneys,", plain["study_id"].nunique(), "studies")
plain_fractions = pd.DataFrame({t: 1 - plain[f"functional_ml_thr{t}"] / plain["total_ml"] for t in thr})
print("Fraction of kidney below threshold (min component 2 mL), percentiles over kidneys:")
print(plain_fractions.describe(percentiles=[.25, .5, .75, .95]).loc[["25%", "50%", "75%", "95%", "max"]].round(3))
plain["fluid_over_functional_pet"] = plain["fluid_mean_pct_ia_ml"] / plain["functional_mean_pct_ia_ml"]
print(plain.sort_values("nonfunctional_fraction")[["study_id", "side", "parenchyma_hu", "total_ml", "functional_ml", "nonfunctional_fraction", "largest_cold_fluid_ml", "fluid_over_functional_pet"]].tail(14).round(2).to_string(index=False))

both = kidneys[kidneys["subject"].isin(plain["subject"]) & kidneys["ct_kidney_enhanced"]]
pairs = plain[plain["subject"].isin(both["subject"])][["subject", "study_id", "side", "total_ml", "nonfunctional_fraction"]].merge(
    both[["subject", "study_id", "side", "total_ml", "nonfunctional_fraction"]], on=["subject", "side"], suffixes=("_plain", "_contrast"))
print("\nSame subject, same side, non-contrast vs contrast CT (different dates):")
print(pairs[["subject", "side", "total_ml_plain", "total_ml_contrast", "nonfunctional_fraction_plain", "nonfunctional_fraction_contrast"]].round(3).to_string(index=False))
print("Median difference in the non-functional fraction:", round((pairs["nonfunctional_fraction_plain"] - pairs["nonfunctional_fraction_contrast"]).median(), 3), "| total volume ratio:", round((pairs["total_ml_plain"] / pairs["total_ml_contrast"]).median(), 3))

kidney_panel([("80b075b681bffb02_2017-07", "left"), ("d98ce837bcc6fb20_2019", "right"), ("ee0d631fa63961f2_2018", "right"), ("e2b34467658ec579", "right"), ("ded97a4f2275ba94_2018-11", "left")], QC / "kidney_noncontrast_cases.png", window=(-60, 120))

**Non-contrast CTs: reading and decision.** **Decision: 10 HU, minimum component 2 mL, kept, with a declared bias.** 23 studies, 46 kidneys. 20 HU is unusable (median 25% of the kidney removed, and 84% at 30 HU, because the parenchyma itself is at 26 HU). At 10 HU the median is 0, the 75th percentile 7.5% and the 95th 17%.

- **True fluid.** Large cold components are found correctly: `ee0d631f` and `e2b34467` have cysts with PET ratios of 0.15-0.22, visible on the panels.
- **False positives.** Kidneys with no cold component have fluid/functional PET ratios up to 0.85 (`18eba3b3`, `079f4e7d`, `ee0d631f` left): those voxels are at least partly parenchyma.
- **Bias.** In 14 same-side pairs from 5 subjects with both CT types, the non-functional fraction is 6 points higher without contrast (total volume ratio 1.04). The dates differ, so this is indicative only. Functional volume is therefore underestimated by about 6% on these CTs.
- **`80b075b6` left** (100-134 mL total, 16-17 mL functional): nearly all at fluid density and without PET uptake, so the flag is right. Its radiological cause is not verified.

## 4. QC flags

In [ ]:
flag_cols = [c for c in summary.columns if c.startswith("flag__") and c not in ("flag__nb09_flags",)]
def flagged(column):
    values = summary[column]
    return values.astype(bool) if values.dtype == bool else values.fillna("").astype(str).str.len().gt(0) & values.fillna("").astype(str).ne("False")
counts = pd.DataFrame({c[6:]: flagged(c).groupby(summary["tracer"]).sum() for c in flag_cols}).T
counts["all"] = counts.sum(axis=1)
print(counts.to_string())
counts.to_csv(QC / "flag_counts.csv")

for flag in ("single_kidney", "small_kidney", "kidney_mostly_nonfunctional", "kidney_cyst"):
    selected = summary[summary[f"flag__{flag}"].fillna("").astype(str).str.len() > 0]
    print(f"\n{flag}: {len(selected)} studies")
    if len(selected) <= 25:
        print(selected[["study_id", "tracer", "kidney_left__total_ml", "kidney_left__functional_ml", "kidney_right__total_ml", "kidney_right__functional_ml", f"flag__{flag}"]].round(0).to_string(index=False))

In [ ]:
special = summary[(summary["flag__single_kidney"].fillna("").astype(str).str.len() > 0) | (summary["flag__small_kidney"].fillna("").astype(str).str.len() > 0)]
n = len(special)
fig, axes = plt.subplots(int(np.ceil(n / 6)), 6, figsize=(14, 2.1 * int(np.ceil(n / 6))), squeeze=False)
for ax in axes.flat:
    ax.set_axis_off()
for ax, sid in zip(axes.flat, special["study_id"]):
    folder = ph.PHANTOMS / sid
    pet = np.asanyarray(nib.load(folder / "pet_pct_ia_ml.nii.gz").dataobj)
    organs = np.asanyarray(nib.load(folder / "organs.nii.gz").dataobj)
    grid = metas[sid]["grid"]
    kidney = np.isin(organs, [1, 2])
    zs = np.where(kidney.any((0, 1)))[0]
    z = slice(max(0, zs[0] - int(60 / grid["spacing_mm"][2])), zs[-1] + int(60 / grid["spacing_mm"][2]))
    mip = pet[:, :, z].max(axis=1)
    ax.imshow(mip.T, cmap="gray_r", origin="lower", vmin=0, vmax=np.percentile(mip, 99.7), aspect=grid["spacing_mm"][2] / grid["spacing_mm"][0])
    ax.contour(kidney[:, :, z].any(axis=1).T, [.5], colors=COHORT, linewidths=.8)
    k = metas[sid]["kidneys"]
    ax.set_title(f"{sid[5:13]} {sid[-10:]}\nL {k['left']['functional_ml']:.0f}  R {k['right']['functional_ml']:.0f} mL", fontsize=8)
fig.suptitle("Single and small kidneys: PET MIP (image left = patient right) with the kidney mask", fontsize=10)
fig.tight_layout(); fig.savefig(QC / "single_small_kidney_mip.png", dpi=130); plt.show()

**Reading.** All 15 flagged studies were viewed (11 18F, 4 68Ga). In none did the flag hide a hot kidney missed by the segmentation: the side declared absent or small shows no or low uptake in the MIP. `kidney_absent_in_segmentation` is 5 studies. These kidneys stay in the phantoms, enter the PBPK with their functional volume, and are excluded from the population statistics below. `organ_cut_by_fov` is 0 for every study, and it covers kidneys, liver, spleen, bladder, heart and aorta only: `81b53d3911e09543` escapes it because its bladder falls wholly outside the CT, and is caught by `compartment_empty` and `pet_outside_body`.

### Salivary glands

All glands are kept in `organs.nii.gz`. `salivary_glands.<gland>.use_for_pbpk` in `meta.json` is true when the gland is present and not truncated: all parotids that reach the top of the CT are out. The group `salivary_glands_pbpk` sums the usable ones only.

In [ ]:
rows = []
for sid, m in metas.items():
    for gland, g in m["salivary_glands"].items():
        rows.append(dict(study_id=sid, tracer=m["tracer"], pbpk_cohort=m["pbpk_cohort"], gland=gland, gland_type=g["gland_type"], present=g["present"],
                         truncated=g["truncated"], use_for_pbpk=g["use_for_pbpk"], volume_ml=g["volume_ml"], total_pct_ia=g["total_pct_ia"], mean_suv=g["mean_suv"]))
glands = pd.DataFrame(rows)
print(glands.groupby(["tracer", "gland"])[["present", "truncated", "use_for_pbpk"]].sum().join(glands.groupby(["tracer", "gland"]).size().rename("n")).to_string())
print()
print(glands[glands["present"]].groupby(["gland_type", "truncated"])[["volume_ml", "total_pct_ia", "mean_suv"]].median().round(2).to_string())
print("\nTruncated submandibular glands (also out of the PBPK fit):")
print(glands.loc[(glands["gland_type"] == "submandibular") & glands["truncated"], ["study_id", "gland", "volume_ml"]].to_string(index=False))
n_pbpk = glands[glands["pbpk_cohort"]].groupby("study_id")["use_for_pbpk"].sum()
print("\nPBPK cohort, usable glands per study:", n_pbpk.value_counts().sort_index().to_dict())
print("Gland with no PET at all:", summary.loc[summary["flag__salivary_no_pet"].fillna("").astype(str).str.len() > 0, ["study_id", "flag__salivary_no_pet"]].to_string(index=False))
low = glands[glands["present"] & ~glands["truncated"] & (glands["mean_suv"] < 2)].merge(summary[["study_id", "lesions_volume_ml", "lesions_total_pct_ia"]], on="study_id")
print(f"\nComplete glands with SUVmean < 2: {len(low)}, in {low['study_id'].nunique()} studies")
print(low.groupby("study_id").agg(tracer=("tracer", "first"), glands=("gland", "size"), lowest_suv=("mean_suv", "min"), tlv_ml=("lesions_volume_ml", "first"), lesions_pct_ia=("lesions_total_pct_ia", "first")).sort_values("tlv_ml", ascending=False).round(2).to_string())


**Reading.** Parotid truncation is the main loss: 233 of 450 present 68Ga parotids are truncated, so only 88 of the 227 PBPK studies have all four glands (one has none, one has a single gland). Truncated glands have the same SUV as complete ones (12.0 vs 11.4), so concentration survives truncation; only volume and total do not.

Low uptake is **not** a QC failure by itself. 30 complete glands in 17 studies have SUVmean below 2. `psma_505ebb983b71c700` (TLV 2240-2860 mL, 25-30 %IA in lesions, all four glands below 2 in both studies) is a clear tumour sink and must stay. The studies with small TLV (several below 30 mL) have no such explanation: they may be gland disease, or PET/CT head mismatch, and were not reviewed. Nine of the 17 studies are 68Ga: review them before the gland fit. `psma_0d2d8c9b29ea6861_2020-04-25` is the extreme case: the left parotid falls wholly on zero PET, and its other three glands have SUV 0.6-1.6, so they are probably off too although the rule removes only the first. It is 18F, so outside the PBPK cohort.

## 5. Validation: %IA per organ at ~1 h, 68Ga cohort

### Literature

Values read from the papers (the numbers below are in the next cell, with the same labels):

- **Green et al.**, Nucl Med Biol 2017;46 (author manuscript, IUPUI ScholarWorks), Table 1: 9 men, biochemical failure, median PSA 0.52 ng/mL, 112.5 MBq, 95.8 kg. %ID by PET Edge VOIs at 15, 60 and 90 min; **a void at about 48 min before the 60 min scan**, so its bladder value is post-void. SUVmean at 60 min in the Results text. This is the only per-organ human %IA table at about 1 h that was found and read.
- **Sandgren et al.**, EJNMMI Phys 2019;6:2: 6 low-risk patients, %IA "kidneys (7%), liver (15%), spleen (2%), and salivary glands (0.5%)". The text gives **no time point** (Fig. 4 has the curves, not read as numbers) and the %IA is median VOI concentration x ICRP 133 mass, so a different estimator.
- **Gaertner et al.**, Oncotarget 2017;8:55094: 135 patients, 68Ga-PSMA-11, 89 +- 22 min, SUVmean by visual tumour load. Glands and kidneys by **50% isocontour**, liver, spleen, mediastinum, bone by spheres, so the values are not whole-organ means: expected above ours for glands and kidneys. Low-load column used; medium and high in the stratified table below.
- **Ferreira et al.**, Cancer Imaging 2019;19:23: 34 men, median 57 min. Reports SUVpeak only: used for the aorta only, not comparable as a mean for the other organs.

**Not found:** whole-organ %IA at 1 h for the GI tract, red marrow, spine, blood pool and rest of body. Demirci et al., Nucl Med Commun 2016 and Pfob et al., EJNMMI 2016 (per-organ data expected) could not be opened in full text: they should be read before the thesis cites this comparison as complete.

### Selection

Cohort: `pbpk_cohort`. Whole-organ comparisons exclude studies where the organ is not complete or not normal: kidneys without single or small kidney, cut organ or nb09 kidney outlier; parotids and submandibular only with both glands of the pair present and not truncated; the cropped study only for bladder and body totals. Lesion voxels are out of the organs (they are in their own label) in all rows.

In [ ]:
literature = pd.DataFrame([
    ("kidneys", "%IA", "Green 2017, 60 min", 14.02, 3.73), ("kidneys", "%IA", "Green 2017, 90 min", 14.21, 4.54), ("kidneys", "%IA", "Sandgren 2019, time not given", 7, np.nan),
    ("liver", "%IA", "Green 2017, 60 min", 6.54, 1.43), ("liver", "%IA", "Green 2017, 90 min", 5.98, 1.39), ("liver", "%IA", "Sandgren 2019, time not given", 15, np.nan),
    ("spleen", "%IA", "Green 2017, 60 min", 1.25, 0.63), ("spleen", "%IA", "Green 2017, 90 min", 1.06, 0.55), ("spleen", "%IA", "Sandgren 2019, time not given", 2, np.nan),
    ("parotid glands", "%IA", "Green 2017, 60 min", 0.87, 0.30), ("parotid glands", "%IA", "Green 2017, 90 min", 0.92, 0.35),
    ("submandibular glands", "%IA", "Green 2017, 60 min", 0.29, 0.19), ("submandibular glands", "%IA", "Green 2017, 90 min", 0.30, 0.19),
    ("salivary glands, all", "%IA", "Sandgren 2019, time not given", 0.5, np.nan),
    ("urinary bladder", "%IA", "Green 2017, 60 min, post-void", 1.22, 0.49), ("urinary bladder", "%IA", "Green 2017, 40 min, pre-void", 2.84, 1.04),
    ("kidneys", "SUVmean", "Green 2017, 60 min", 30.1, 6.6), ("liver", "SUVmean", "Green 2017, 60 min", 3.3, 0.6), ("spleen", "SUVmean", "Green 2017, 60 min", 5.2, 2.5), ("parotid glands", "SUVmean", "Green 2017, 60 min", 9.4, 2.0),
    ("kidneys", "SUVmean", "Gaertner 2017, low load", 29.9, 9.59), ("liver", "SUVmean", "Gaertner 2017, low load", 4.34, 1.33), ("spleen", "SUVmean", "Gaertner 2017, low load", 6.95, 2.92),
    ("parotid glands", "SUVmean", "Gaertner 2017, low load", 11.69, 3.03), ("submandibular glands", "SUVmean", "Gaertner 2017, low load", 12.93, 3.40),
    ("blood pool", "SUVmean", "Gaertner 2017, mediastinum, low load", 1.06, 0.19), ("blood pool", "SUVmean", "Ferreira 2019, aorta SUVpeak", 1.3, 0.29),
    ("spine L2-L4", "SUVmean", "Gaertner 2017, lumbar and sacral bone, low load", 0.91, 0.39),
], columns=["organ", "quantity", "source", "mean", "sd"])

cohort = summary[summary["pbpk_cohort"]].copy()
empty = lambda name: cohort[f"flag__{name}"].fillna("").astype(str).str.len() == 0
normal_kidneys = empty("single_kidney") & empty("small_kidney") & empty("kidney_outlier") & ~cohort["flag__organ_cut_by_fov"].fillna("").str.contains("kidney")
complete_body = ~cohort["flag__cropped_not_whole_body"]
both_ok = lambda a, b: cohort[f"{a}__use_for_pbpk"] & cohort[f"{b}__use_for_pbpk"]
parotids = both_ok("parotid_gland_left", "parotid_gland_right")
submandibular = both_ok("submandibular_gland_left", "submandibular_gland_right")
all_glands = parotids & submandibular

rows = [
    ("kidneys", "kidneys_total", normal_kidneys), ("kidneys, grown 10 mm", None, normal_kidneys),
    ("liver", "liver", ~cohort["flag__organ_cut_by_fov"].fillna("").str.contains("liver")), ("spleen", "spleen", ~cohort["flag__organ_cut_by_fov"].fillna("").str.contains("spleen")),
    ("parotid glands", "parotid_glands", parotids), ("submandibular glands", "submandibular_glands", submandibular), ("salivary glands, all", "salivary_glands_all", all_glands),
    ("urinary bladder", "urinary_bladder", complete_body & (cohort["urinary_bladder__volume_ml"] > 1)), ("GI tract", "gi_tract", complete_body), ("blood pool", "blood_pool", ~cohort["flag__blood_pool_small"]),
    ("spine L2-L4", "spine_L2_L4", cohort["flag__marrow_L2_L4_incomplete"].fillna("").eq("")), ("whole spine", "spine_whole", complete_body), ("rest of body", "rest_of_body", complete_body), ("lesions", "lesions", complete_body),
]
out, values = [], {}
for name, key, select in rows:
    sub = cohort[select]
    if key is None:
        pct, suv = sub["kidneys_total__dilated_10mm_total_pct_ia"], None
    elif key == "lesions":
        pct, suv = sub["lesions_total_pct_ia"], None
    else:
        pct, suv = sub[f"{key}__total_pct_ia"], sub[f"{key}__mean_suv"]
    values[name] = dict(pct=pct.dropna(), suv=None if suv is None else suv.dropna())
    entry = dict(organ=name, n=len(pct.dropna()), pct_median=pct.median(), pct_q25=pct.quantile(.25), pct_q75=pct.quantile(.75))
    if suv is not None:
        entry.update(suv_median=suv.median(), suv_q25=suv.quantile(.25), suv_q75=suv.quantile(.75))
    if key not in (None, "lesions"):
        entry["volume_ml_median"] = sub[f"{key}__volume_ml"].median()
    out.append(entry)
result = pd.DataFrame(out)
result.to_csv(QC / "biodistribution_68ga.csv", index=False)
literature.to_csv(QC / "biodistribution_literature.csv", index=False)
print(f"68Ga cohort: {len(cohort)} studies, {cohort['subject'].nunique()} subjects; uptake median {cohort['uptake_time_min'].median():.0f} min (IQR {cohort['uptake_time_min'].quantile(.25):.0f}-{cohort['uptake_time_min'].quantile(.75):.0f}); weight median {cohort['weight_kg'].median():.0f} kg")
print(result.round(2).to_string(index=False))
print()
print(literature.to_string(index=False))

In [ ]:
def biodistribution_plot(quantity, key, xlabel, title, path, order):
    fig, ax = plt.subplots(figsize=(8, .55 * len(order) + 1.4))
    for y, name in enumerate(order):
        data_values = values[name][key]
        ax.boxplot(data_values, positions=[y], orientation="horizontal", widths=.45, showfliers=False, patch_artist=True,
                   boxprops=dict(facecolor="#cde2fb", edgecolor=COHORT), medianprops=dict(color=COHORT, lw=2), whiskerprops=dict(color=COHORT), capprops=dict(color=COHORT))
        lit = literature[(literature["organ"] == name.replace(", grown 10 mm", "")) & (literature["quantity"] == quantity)]
        for offset, (_, row) in zip(np.linspace(-.32, .32, max(len(lit), 1)) if len(lit) > 1 else [-.32], lit.iterrows()):
            ax.errorbar(row["mean"], y + offset, xerr=None if np.isnan(row["sd"]) else row["sd"], fmt="o", ms=5, color=LITERATURE, capsize=2, lw=1)
    ax.set_yticks(range(len(order)), order); ax.invert_yaxis(); ax.set_xscale("log")
    ax.set(xlabel=xlabel, title=title); ax.grid(axis="x", color="#e5e4e0", lw=.6); ax.set_axisbelow(True)
    ax.legend(handles=[Patch(facecolor="#cde2fb", edgecolor=COHORT, label="68Ga cohort: median, IQR, 1.5 IQR whiskers"),
                       plt.Line2D([], [], marker="o", color=LITERATURE, ls="", label="Literature: mean +- SD (no bar if not given)")], frameon=False, loc="upper center", bbox_to_anchor=(.5, -.09 - .3 / len(order)), ncol=2, fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.show()

biodistribution_plot("%IA", "pct", "%IA, decay-corrected, per organ", "Whole-organ %IA at about 1 h: 68Ga cohort and published values", QC / "biodistribution_pct_ia.png",
                     ["kidneys", "kidneys, grown 10 mm", "liver", "spleen", "parotid glands", "submandibular glands", "salivary glands, all", "urinary bladder"])
biodistribution_plot("SUVmean", "suv", "SUVmean", "Whole-organ SUVmean at about 1 h: 68Ga cohort and published values", QC / "biodistribution_suv.png",
                     ["kidneys", "liver", "spleen", "parotid glands", "submandibular glands", "blood pool", "spine L2-L4"])

In [ ]:
tertiles = cohort["lesions_volume_ml"].where(cohort["n_lesions"] > 0)
groups = pd.cut(tertiles, tertiles.quantile([0, 1/3, 2/3, 1]).values, include_lowest=True, labels=["low TLV", "mid TLV", "high TLV"]).astype(object).fillna("no lesion")
strat = {}
for name, key, select in [("kidneys", "kidneys_total", normal_kidneys), ("liver", "liver", pd.Series(True, index=cohort.index)), ("spleen", "spleen", pd.Series(True, index=cohort.index)), ("parotid", "parotid_glands", parotids), ("submandibular", "submandibular_glands", submandibular)]:
    strat[name + " SUV"] = cohort[select].groupby(groups[select])[f"{key}__mean_suv"].median()
    strat[name + " %IA"] = cohort[select].groupby(groups[select])[f"{key}__total_pct_ia"].median()
strat = pd.DataFrame(strat).loc[["no lesion", "low TLV", "mid TLV", "high TLV"]]
strat.insert(0, "n", groups.value_counts())
print(strat.round(2).to_string())
print("\nGaertner 2017 SUVmean low / medium / high load: kidneys 29.9/24.4/16.6, liver 4.34/4.19/3.27, spleen 6.95/6.47/5.66, parotid 11.69/8.02/6.62, submandibular 12.93/8.46/8.24")
strat.to_csv(QC / "biodistribution_by_tlv.csv")

kid = cohort[normal_kidneys]
print("\nKidney %IA grown 10 mm / CT mask: median", round((kid["kidneys_total__dilated_10mm_total_pct_ia"] / kid["kidneys_total__total_pct_ia"]).median(), 2))
print("Functional / total kidney volume, median:", round((kid["kidney_functional__volume_ml"] / kid["kidneys_total__volume_ml"]).median(), 3))
print("Functional / total kidney %IA, median:", round((kid["kidney_functional__total_pct_ia"] / kid["kidneys_total__total_pct_ia"]).median(), 3))

In [ ]:
green = {"kidneys": (14.02, 30.1), "liver": (6.54, 3.3), "spleen": (1.25, 5.2), "parotid glands": (0.87, 9.4)}
GREEN_WEIGHT_G = 95.8e3
rows = []
for name, (pct, suv) in green.items():
    row = result.set_index("organ").loc[name]
    rows.append(dict(organ=name, green_pct_ia=pct, cohort_pct_ia=row["pct_median"], green_suv=suv, cohort_suv=row["suv_median"], green_implied_ml=pct * GREEN_WEIGHT_G / (100 * suv), cohort_ml=row["volume_ml_median"]))
decomposition = pd.DataFrame(rows)
decomposition["implied_over_cohort_ml"] = decomposition["green_implied_ml"] / decomposition["cohort_ml"]
print(decomposition.round(2).to_string(index=False))
decomposition.to_csv(QC / "biodistribution_green_decomposition.csv", index=False)


**Reading.** **The cohort is consistent with the literature when concentration is compared; the %IA gaps come mostly from how volumes are defined.**

- **SUVmean agrees for every organ with a comparator.** Kidneys 25.8 vs 30 (below, because of the misregistration and partial volume below), liver 4.14 vs 3.3-4.3, spleen 5.7 vs 5.2-7.0, parotid 9.8 vs 9.4-11.7, submandibular 11.4 vs 12.9 (Gaertner's 50% isocontour is expected higher), blood pool 1.26 vs 1.06-1.3, L2-L4 0.92 vs 0.91.
- **%IA differs where the VOI differs.** Green's %IA and SUV imply a VOI volume through %IA = SUV x V / W. This is an estimate from ratios of means, at their 95.8 kg against our 80 kg. Their implied volume is 1.2 times ours for liver and spleen, 1.4 for kidneys and **3.4 for parotids** (89 vs 26 mL).
- **Parotids: 0.30 vs 0.87 %IA, with the same SUV.** The gap is the volume. Our 12.4 mL per gland is below the 19-25 mL of Okamoto and Scarpa (35-50% lower, the underestimation known from nb09), and Green's Edge VOI at 44 mL per gland is well above those references, so it presumably includes spill-out. Neither %IA is a clean whole-gland value. Fit the glands on concentration, or correct the volume, not on this %IA.
- **Kidneys: 10.3 %IA in the CT mask, 14.6 when grown by 10 mm (factor 1.41), Green 14.0.** The CT mask misses activity because of PET/CT misregistration and partial volume. The factor matches Green's implied volume ratio (1.38), which supports the reading. For the PBPK kidney total, **use the grown mask** (`dilated_10mm_total_pct_ia`, grown into rest of body only). The contamination risk from neighbours was not measured.
- **Liver:** 8.2 %IA lies between Green (6.5) and Sandgren (15). Sandgren's estimator (median concentration x ICRP mass, no stated time) is not comparable, and neither is its kidney value of 7 %IA.
- **Bladder:** 1.31 %IA vs 1.22 post-void and 2.84 pre-void. The voiding status of our cohort is not known, so the value cannot be interpreted further.
- **Tumour sink.** Kidney SUV falls from 29 to 20 and parotid from 11-12 to 8 in the highest TLV tertile, with the same direction as Gaertner (29.9 -> 16.6, 11.7 -> 6.6). Liver and spleen change little in our data (4.3 -> 3.9, about 6). This dependence on TLV belongs in the PBPK population.
- **Without a comparator:** GI tract 4.7, whole spine 1.2 and rest of body 41.9 %IA are not validated by any whole-organ value found. Demirci 2016 and Pfob 2016 were not read.
- **Functional vs total kidney:** the median ratio is 1.00 in both volume and %IA, so the choice only matters for flagged kidneys.

## 6. Coronal overlays, random sample

Random 68Ga studies, fixed seed; left CT with compartments, right PET MIP. The slice is the coronal plane through the kidneys.

In [ ]:
colors = {1: "#2a78d6", 2: "#e87ba4", 3: "#eb6834", 4: "#6250d6", 5: "#1baf7a", 6: "#1baf7a", 7: "#008300", 8: "#008300", 9: "#eda100", 10: "#e34948", 11: "#00b7c3", 12: "#8a5a2b", 13: "#b8b8b0", 15: "#ff00ff"}
names = {1: "kidney, functional", 2: "kidney, fluid", 3: "liver", 4: "spleen", 5: "parotid", 7: "submandibular", 9: "bladder", 10: "GI tract", 11: "blood pool", 12: "L2-L4", 13: "other spine", 15: "lesion"}
palette = ListedColormap([(0, 0, 0, 0)] + [colors.get(i, (0, 0, 0, 0)) for i in range(1, 16)])

def coronal(sid, ax_ct, ax_pet):
    folder = ph.PHANTOMS / sid
    ct = np.asanyarray(nib.load(folder / "ct.nii.gz").dataobj)
    organs = np.asanyarray(nib.load(folder / "organs.nii.gz").dataobj)
    pet = np.asanyarray(nib.load(folder / "pet_pct_ia_ml.nii.gz").dataobj)
    spacing = metas[sid]["grid"]["spacing_mm"]
    aspect = spacing[2] / spacing[0]
    kidney = np.isin(organs, [1, 2])
    y = int(np.median(np.where(kidney)[1])) if kidney.any() else organs.shape[1] // 2
    ax_ct.imshow(ct[:, y, :].T, cmap="gray", vmin=-250, vmax=350, origin="lower", aspect=aspect)
    shown = organs[:, y, :].T.astype(float).copy()
    shown[shown == 14] = 0
    ax_ct.imshow(shown, cmap=palette, vmin=0, vmax=15, origin="lower", aspect=aspect, alpha=.55, interpolation="nearest")
    ax_ct.contour((organs[:, y, :] > 0).T, [.5], colors="#eda100", linewidths=.5)
    mip = pet.max(axis=1)
    ax_pet.imshow(mip.T, cmap="gray_r", vmin=0, vmax=np.percentile(mip[mip > 0], 99.5) * .5, origin="lower", aspect=aspect)
    m = metas[sid]
    ax_ct.set_title(f"{sid[5:13]} {sid[-10:]}, {m['lesions']['n']} lesions", fontsize=7.5, loc="left")
    for ax in (ax_ct, ax_pet):
        ax.set_axis_off()

rng = np.random.default_rng(20261103)
sample = list(rng.choice(cohort["study_id"], 10, replace=False))
fig, axes = plt.subplots(2, 10, figsize=(16, 7.8))
for i, sid in enumerate(sample):
    r, c = (0, i * 2) if i < 5 else (1, (i - 5) * 2)
    coronal(sid, axes[r, c], axes[r, c + 1])
fig.legend(handles=[Patch(color=colors[k], label=v) for k, v in names.items()], ncol=len(names), frameon=False, fontsize=8, loc="lower center")
fig.tight_layout(rect=(0, .03, 1, 1)); fig.savefig(QC / "overlays_random_68ga.png", dpi=130); plt.show()
print(sample)

## 7. Outputs and limitations

Outputs: `data/processed/psma/phantoms/<study_id>/`, `phantoms/labelmap.json`, `data/processed/psma/phantoms_summary.csv` (one row per phantom), `data/processed/psma/phantoms_qc/` (figures, flag counts, biodistribution tables).

**Status.** 588 of 597 studies built (9 excluded by the list of `src/psma_exclusions.py`), 10.5 GB. `psma_3959d1c381a5bcd6_2015-11-12` is cropped at k >= 84: no bladder, not usable for whole-body totals.

**Limitations.**
- PET at CT masks: PET and CT are not motion-matched. Kidneys lose about 30% of their activity in the CT mask and a moved head can put glands outside the PET (`psma_0d2d8c9b29ea6861_2020-04-25`, flag `salivary_no_pet`). Only that total failure is detected.
- Glands: under-segmented by about 35-50% against published volumes, at PET resolution; more than half of the 68Ga parotids are truncated. The parotid model has no per-structure validation in its required reference (Walter et al., Cancers 2024: submandibular glands and thyroid only), so only our QC supports it.
- Non-contrast kidneys: about 6 points of false fluid.
- Scanner effect on the FOV %IA not resolved; weights and activities come from headers.
- Red-marrow surrogate: L2-L4 includes cortical bone and lesion voxels go to label 15. It is a surrogate only.
- Validation is against SUV from three small or differently defined series; only Green 2017 gives whole-organ %IA near 1 h. Green's value is from 9 patients and read from an author manuscript.
- `build_log_last_run.csv` reflects the last run only. The first build took about 5 process-hours (mean 31 s per study, 6 processes, about 50 min). The 12 studies affected by the slice repair and the failed study were built after it; `meta.json` fields added afterwards (`salivary_no_pet`, `slices_without_body`, `pct_ia_in_slices_without_body`) and the corrected `compartment_empty` were written in place for the rest and match a rebuild on the samples checked.